In [1]:
import requests
import json

url = "https://api.binance.com/api/v3/ticker/24hr"

response = requests.get(url)

data = response.json()

# First 10 popular coins
popular_coins = [
    coin for coin in data
    if coin["symbol"] in [
        "BTCUSDT","ETHUSDT","BNBUSDT","XRPUSDT","ADAUSDT",
        "DOGEUSDT","SOLUSDT","DOTUSDT","MATICUSDT","LTCUSDT"
    ]
]

with open("crypto_data.json", "w") as file:
    json.dump(popular_coins, file, indent=4)

print("crypto_data.json created successfully")

crypto_data.json created successfully


In [2]:
import json

with open("crypto_data.json", "r") as file:
    data = json.load(file)

def find_most_volatile_coin(data):
    
    most_volatile = max(
        data,
        key=lambda coin: abs(float(coin["priceChangePercent"]))
    )
    
    return most_volatile["symbol"]

print("Most Volatile Coin:", find_most_volatile_coin(data))

Most Volatile Coin: BTCUSDT


In [3]:
import json

with open("crypto_data.json", "r") as file:
    data = json.load(file)

prices = [float(coin["lastPrice"]) for coin in data]

average_price = sum(prices) / len(prices)

print("Average Price:", round(average_price, 2))

print("\nCoins Below Average Price:")

for coin in data:
    if float(coin["lastPrice"]) < average_price:
        print(
            coin["symbol"],
            "-",
            coin["lastPrice"]
        )

Average Price: 6415.85

Coins Below Average Price:
ETHUSDT - 1646.69000000
BNBUSDT - 589.58000000
LTCUSDT - 42.30000000
ADAUSDT - 0.16630000
XRPUSDT - 1.14580000
MATICUSDT - 0.37940000
DOGEUSDT - 0.08492000
SOLUSDT - 65.05000000
DOTUSDT - 0.95400000


In [4]:
import json

with open("crypto_data.json", "r") as file:
    data = json.load(file)

def rank_coins_by_volume(data):

    ranked = sorted(
        data,
        key=lambda coin: float(coin["volume"]),
        reverse=True
    )

    print("Top 5 Coins By Volume\n")

    for rank, coin in enumerate(ranked[:5], start=1):
        print(
            f"{rank}. {coin['symbol']} - Volume: {coin['volume']}"
        )

rank_coins_by_volume(data)

Top 5 Coins By Volume

1. DOGEUSDT - Volume: 579545615.00000000
2. ADAUSDT - Volume: 172170121.10000000
3. XRPUSDT - Volume: 106057203.20000000
4. DOTUSDT - Volume: 4844801.08000000
5. SOLUSDT - Volume: 3056360.76500000


In [5]:
!pip install schedule

In [ ]:
import requests
import json
import time
import schedule

def fetch_crypto_data():

    url = "https://api.binance.com/api/v3/ticker/24hr"

    retries = 5

    for attempt in range(retries):

        response = requests.get(url)

        if response.status_code == 200:

            data = response.json()

            with open("crypto_data.json", "w") as file:
                json.dump(data, file, indent=4)

            print("Data Updated Successfully")
            return

        elif response.status_code == 429:

            wait_time = 2 ** attempt

            print(
                f"Rate Limit Hit. Retrying in {wait_time} seconds..."
            )

            time.sleep(wait_time)

        else:

            print(
                "API Error:",
                response.status_code
            )

            return

schedule.every(1).hours.do(fetch_crypto_data)

print("Scheduler Started...")

while True:
    schedule.run_pending()
    time.sleep(1)

Scheduler Started...
